In [1]:
!pip install -q -U google-genai pydantic requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.1/487.1 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 18.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import json
from google import genai
from google.colab import userdata
from pydantic import BaseModel, Field
from typing import List

# 1. Connect to Gemini using your Colab Secret
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

# 2. Define schema for MITRE extraction
class TechniqueExtraction(BaseModel):
    technique_id: str = Field(description="MITRE ATT&CK ID, e.g., T1566.002 or T1059.001")
    technique_name: str = Field(description="Official name of the MITRE technique")
    tactic: str = Field(description="Primary tactic (e.g., Initial Access, Persistence)")
    evidence_quote: str = Field(description="Exact sentence from the text proving this technique")
    confidence: float = Field(description="Confidence score between 0.0 and 1.0")

class CTIReport(BaseModel):
    threat_actor: str = Field(description="Threat group name or campaign identifier")
    summary: str = Field(description="Concise operational summary of the attack chain")
    techniques: List[TechniqueExtraction]

# 3. Threat advisory input text
report_text = """
The threat group known as Scattered Spider executed targeted social engineering campaigns.
Attackers sent SMS phishing links imitating Okta login portals to steal employee credentials.
They then enrolled new MFA hardware tokens to establish persistence on compromised accounts.
After gaining initial access, the group used PowerShell scripts to discover Active Directory
users and installed AnyDesk software to maintain remote command and control.
"""

# 4. Extract techniques using Gemini
print("[*] Analyzing threat report with Gemini...")
prompt = f"Analyze this cyber threat report and extract all MITRE ATT&CK techniques:\n\n{report_text}"

response = client.models.generate_content(
    model='gemini-3.8-flash',
    contents=prompt,
    config={
        'response_mime_type': 'application/json',
        'response_schema': CTIReport,
    },
)

data = json.loads(response.text)

# 5. Build and save the ATT&CK Navigator Layer
navigator_layer = {
    "name": f"{data['threat_actor']} - Automated Analysis",
    "versions": {"attack": "15", "navigator": "5.0.0", "layer": "4.5"},
    "domain": "enterprise-attack",
    "description": data["summary"],
    "techniques": [
        {
            "techniqueID": t["technique_id"],
            "score": int(t["confidence"] * 100),
            "comment": f"[{t['tactic']}] Evidence: {t['evidence_quote']}",
            "enabled": True
        }
        for t in data["techniques"]
    ],
    "gradient": {"colors": ["#ffebee", "#c62828"], "minValue": 0, "maxValue": 100}
}

with open("attack_layer.json", "w") as f:
    json.dump(navigator_layer, f, indent=2)

print(f"[+] Success! Generated 'attack_layer.json' with {len(data['techniques'])} mapped techniques.")

[*] Analyzing threat report with Gemini...
[+] Success! Generated 'attack_layer.json' with 5 mapped techniques.
